# Exploratory Data Analysis

This notebook produces reproducible input-data and generated-result diagnostics for the first part of the Results section when an explicit execution mode is selected. All figures use a colourblind-safe palette and are saved in both PNG and PDF under `analysis/figures/`; tables are exported as booktabs-compatible `.tex` files under `analysis/tables/`.

## Input files and their intended use

| File | Content | Usage |
|---|---|---|
| `results.csv` | Full history of international matches (Kaggle). Columns: `date, home_team, away_team, home_score, away_score, tournament, city, country, neutral`. Includes scheduled 2026 World Cup matches (null scores until played). | Main training source (`prepare_cycle_data`) and extraction of known 2026 results in `sim_2026.py`. |
| `shootouts.csv` | Penalty shootout results (Kaggle). Columns: `date, home_team, away_team, winner, first_shooter`. | Downloaded by `data_pipeline()` with `results.csv`; available for use in the model. |
| `goalscorers.csv` | Goal scorers per match (Kaggle). Columns: `date, home_team, away_team, team, scorer, minute, own_goal, penalty`. | Downloaded by `data_pipeline()`, but **not consumed by any active script** (`load_data()`, the only function that returns it, is not called by the pipeline). |
| `world_cup_results.csv` | Official 2026 World Cup calendar. Columns: `stage, group, home_team, away_team, home_real, away_real, date`. Team names in Portuguese. Includes empty rows for knockout stages. | Reference for home/away orientation and group of each group-stage match; described as read by `_load_schedule_orientations()` in project documentation. |
| `raw/fifa_ranking_2010.csv` | FIFA ranking at the end of the 2010 World Cup. Columns: `date, semester, rank, team, acronym, total.points, previous.points, diff.points`. | Not referenced by any current script (no 2010 simulation exists). |
| `raw/fifa_ranking_2014.csv` | FIFA ranking at the end of the 2014 World Cup. Same columns. | Bayesian prior for the 2014 cycle in backtesting (`validate.py`). |
| `raw/fifa_ranking_2018.csv` | FIFA ranking at the end of the 2018 World Cup. Same columns. | Prior for the 2022 cycle in backtesting (`validate.py`). |
| `raw/fifa_ranking_2022.csv` | FIFA ranking at the end of the 2022 World Cup. Same columns. | Prior for the 2026 model in `train_2026.py` (`load_ranking_priors`). |
| `raw/jogos_2014.csv`, `raw/jogos_2018.csv`, `raw/jogos_2022.csv` | Matches of the 2014, 2018 and 2022 World Cups. Same columns as `results.csv`. | Test sets for backtesting in `validate.py`. |

**Repository audit note.** The table describes the intended input inventory. In this checkout, `data/world_cup_results.csv` is absent and the active `_load_schedule_orientations()` implementation reads `data/results.csv`; the notebook's audit reports that discrepancy. Only results known up to a cutoff are fixed; later real matches remain simulated in a snapshot.

In [ ]:
# Configuration: no absolute paths are used.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
ANALYSIS_ROOT = PROJECT_ROOT / 'analysis'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODE = 'FULL'  # Set FAST for input EDA or FULL after Notebook 1 has created snapshots.
assert MODE in {'PLAN', 'FAST', 'FULL'}

from analysis.audit_pipeline import data_inventory, run_exploratory_data_analysis

try:
    from IPython.display import display
except ImportError:
    display = print

## A. Schema, missingness and duplicate audit

In [2]:
if MODE in {'FAST', 'FULL'}:
    inventory = data_inventory(PROJECT_ROOT)
    display(inventory)
else:
    print('PLAN mode: no input audit was run.')

,file,path,exists,expected_columns,n_rows,missing_expected_columns,read_error
0,results.csv,data/results.csv,True,"date, home_team, away_team, home_score, away_s...",11648.0,,
1,shootouts.csv,data/shootouts.csv,True,"date, home_team, away_team, winner, first_shooter",684.0,,
2,goalscorers.csv,data/goalscorers.csv,True,"date, home_team, away_team, team, scorer, minu...",47867.0,,
3,world_cup_results.csv (documented input),data/world_cup_results.csv,False,"stage, group, home_team, away_team, home_real,...",NaN,,
4,public_calendar.csv,docs/csv/vis/calendario.csv,True,"fase, rodada, grupo, data, mandante, visitante",104.0,,
5,fifa_ranking_2010.csv,data/raw/fifa_ranking_2010.csv,True,"date, semester, rank, team, acronym",207.0,,
6,fifa_ranking_2014.csv,data/raw/fifa_ranking_2014.csv,True,"date, semester, rank, team, acronym",209.0,,
7,fifa_ranking_2018.csv,data/raw/fifa_ranking_2018.csv,True,"date, semester, rank, team, acronym",211.0,,
8,fifa_ranking_2022.csv,data/raw/fifa_ranking_2022.csv,True,"date, semester, rank, team, acronym",211.0,,
9,jogos_2014.csv,data/raw/jogos_2014.csv,True,"date, home_team, away_team, home_score, away_s...",64.0,,


The EDA uses `prepare_cycle_data()` from the project for the training-window definition and weights. It then reports match volume, tournament composition, team coverage, neutral share, goal distribution and Poisson diagnostics, draw rates, observed home/away differences, and the combined time-decay/tournament weights.

## B. Generate publication figures and tables

In [3]:
# This runs on actual local input data only after an explicit FAST or FULL selection.
if MODE in {'FAST', 'FULL'}:
    eda_manifest = run_exploratory_data_analysis(PROJECT_ROOT)
    eda_manifest
else:
    print('PLAN mode: no EDA figures or tables were generated.')

Dados preparados: 3387 jogos e 190 seleções identificadas.
Sucesso! Posições do ranking /home/iara/Github/WorldCup2026/data/raw/fifa_ranking_2022.csv carregadas e normalizadas.


When FULL snapshot products are available, the same call additionally creates: attack/defence and ranking-prior diagnostics; stage-probability heatmaps and trajectories; title concentration; draw probability versus expected-goal mismatch; selected and hypothetical scoreline heatmaps; and probability-mass consistency checks. The generated `figure_manifest.md` maps every delivered item to an existing or clearly proposed LaTeX label.

In [5]:
# Show the manifest only when this notebook explicitly generated artifacts.
if MODE in {'FAST', 'FULL'}:
    manifest_path = ANALYSIS_ROOT / 'outputs_manifest.md'
    print(manifest_path.read_text(encoding='utf-8'))
else:
    print('PLAN mode: the template manifest remains unchanged.')


# Figure and Table Manifest

All paths are relative to `analysis/`. The manuscript labels identify the intended Results or Appendix placement.

## Figures

| Artifact | Files | LaTeX label | Status / use |
|---|---|---|---|
| `predictive_pipeline` | `figures/predictive_pipeline.png` | `\label{fig:pipeline}` | Methods: code-derived flow from inputs through posterior simulation and public outputs. |
| `hierarchical_prior` | `figures/hierarchical_prior.png` | `\label{fig:hierarchical-prior}` | Methods: direct rendering of the selected ranking-informed Stan hierarchy. |
| `time_decay` | `figures/time_decay.png` | `\label{fig:time-decay}` | Methods: exact exponential temporal factor in the implemented match weight. |
| `bracket_structure` | `figures/bracket_structure.png` | `\label{fig:bracket-structure}` | Methods: 48-team simulator topology derived from repository constants. |
| `model_selection_brier` | `figures/model_selection_brier.png` | `\label{fig:model-selection-brier}` | Main Resu

## Interpretation guardrails

Figures show data and saved forecasts; they do not establish model superiority. Forecast-dependent panels are deliberately omitted in FAST mode rather than filled with illustrative values. The input-data EDA is fully reproducible in FAST mode because it never needs posterior sampling. PLAN mode is intentionally inert.